In [1]:
import numpy as np
import matplotlib.pyplot as plt


def read_latency_data(file_path, ignore):
    with open(file_path, "r") as file:
        latencies = [int(line.strip()) for line in file if line.strip().isdigit()]
    n_ignore = int(len(latencies) * ignore)
    latencies = latencies[n_ignore:]
    return latencies


def get_pctl(data, pctl):
    data = np.sort(data)
    tail = np.percentile(data, pctl).astype(int)
    return tail

In [2]:
rps_values = [1500, 1550, 1600, 1650, 1700, 1750, 1800, 1850, 1900, 1950, 2000]
# rps_values = [1500, 1600, 1700, 1800, 1900, 2000]

datas: dict = {}

for pctl in [50, 90, 95, 99, 99.9]:
    data = {
        "rps": [],
        "fcfs": [],
        "masa": [],
        "relative": [],
    }
    for rps in rps_values:
        data["rps"].append(rps)

        file_path = f"r{rps}-fcfs.csv"
        tail = get_pctl(read_latency_data(file_path, ignore=0.2), pctl)
        data["fcfs"].append(tail)

        file_path = f"r{rps}-masa.csv"
        tail = get_pctl(read_latency_data(file_path, ignore=0.2), pctl)
        data["masa"].append(tail)

        data["relative"].append((data["fcfs"][-1] - data["masa"][-1]) / data["fcfs"][-1] * 100)
    datas[pctl] = data

print(datas)

{50: {'rps': [1500, 1550, 1600, 1650, 1700, 1750, 1800, 1850, 1900, 1950, 2000], 'fcfs': [95720, 107588, 120767, 136948, 162464, 201949, 247819, 343452, 580285, 773196, 738900], 'masa': [100684, 112830, 126458, 142059, 169929, 209315, 257916, 359033, 611401, 792750, 787855], 'relative': [-5.185959047221061, -4.872290590028628, -4.712380037593051, -3.7320734877471744, -4.5948640929682885, -3.6474555457070847, -4.074344582134541, -4.536587354273668, -5.362192715648345, -2.5289835953626247, -6.625389091893355]}, 90: {'rps': [1500, 1550, 1600, 1650, 1700, 1750, 1800, 1850, 1900, 1950, 2000], 'fcfs': [162936, 182680, 203268, 226826, 271722, 330511, 406425, 567577, 931224, 1159306, 1273018], 'masa': [147052, 160050, 181685, 200094, 235491, 293985, 365700, 506329, 847940, 986927, 994244], 'relative': [9.748612952324839, 12.387781913728926, 10.618001849774682, 11.785245077724776, 13.33384856581359, 11.051371966439847, 10.020298948145415, 10.791134947328732, 8.943498019810486, 14.86915447690256

In [3]:
import pandas as pd

for pctl in [90, 95, 99, 99.9]:
    data = datas[pctl]
    data["decrease"] = data["relative"]
    data["improve"] = [1.0 / (1.0 - x / 100) for x in data["relative"]]

    data["decrease"] = [f"{round(x, 2)}%" for x in data["decrease"]]
    data["improve"] = [f"{round(x, 2)}%" for x in data["improve"]]

    df = pd.DataFrame(data)
    df = df[["rps", "decrease", "improve"]]
    print(f"tail: {pctl}%")
    print(df)
    print()

tail: 90%
     rps decrease improve
0   1500    9.75%   1.11%
1   1550   12.39%   1.14%
2   1600   10.62%   1.12%
3   1650   11.79%   1.13%
4   1700   13.33%   1.15%
5   1750   11.05%   1.12%
6   1800   10.02%   1.11%
7   1850   10.79%   1.12%
8   1900    8.94%    1.1%
9   1950   14.87%   1.17%
10  2000    21.9%   1.28%

tail: 95%
     rps decrease improve
0   1500   12.58%   1.14%
1   1550   15.35%   1.18%
2   1600   13.54%   1.16%
3   1650   14.88%   1.17%
4   1700   16.65%    1.2%
5   1750   13.79%   1.16%
6   1800   11.31%   1.13%
7   1850   15.12%   1.18%
8   1900   13.44%   1.16%
9   1950   18.67%   1.23%
10  2000   29.46%   1.42%

tail: 99%
     rps decrease improve
0   1500   17.06%   1.21%
1   1550   20.65%   1.26%
2   1600   18.47%   1.23%
3   1650   15.98%   1.19%
4   1700   22.06%   1.28%
5   1750   14.63%   1.17%
6   1800     9.2%    1.1%
7   1850   20.74%   1.26%
8   1900   22.62%   1.29%
9   1950   21.82%   1.28%
10  2000   36.69%   1.58%

tail: 99.9%
     rps decrease i

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams["font.family"] = "Roboto"

plt.rcParams.update(
    {
        "font.size": 20,  # Sets the base default font size
        "axes.labelsize": 20,  # Font size for x and y labels
        "axes.titlesize": 20,  # Font size for plot title
        "xtick.labelsize": 20,  # Font size for x-axis tick labels
        "ytick.labelsize": 20,  # Font size for y-axis tick labels
        "legend.fontsize": 20,  # Font size for legend
    }
)

data = raw

for key in ["fcfs", "masa"]:
    data[key] = [x / 1e3 for x in data[key]]

data["diff"] = []
for i in range(len(data["rps"])):
    data["diff"].append(data["masa"][i] - data["fcfs"][i])

print(data)

# print(data)

# Setting the positions of the bars on the x-axis
x = np.arange(len(data["rps"]))  # the label locations
width = 0.3  # the width of the bars

fig, ax = plt.subplots(figsize=(10, 6))
rects1 = ax.bar(
    x - width / 2,
    data["fcfs"],
    width,
    label="FCFS",
    color="#C25759",
)
rects2 = ax.bar(
    x + width / 2,
    data["masa"],
    width,
    label="Masa",
    color="#599CB4",
)

# Annotate the bars with the diff
# for i, diff in enumerate(data["diff"]):
#     if diff > 0:
#         text = f"+{diff:.1f}"
#     else:
#         text = f"{diff:.1f}"
#     ax.text(
#         x[i] + width / 2,
#         max(data["fcfs"][i], data["masa"][i]) + 0.2,
#         text,
#         ha="center",
#         va="bottom",
#         color="black",
#         fontsize=18,
#     )

# Add some text for labels, title and custom x-axis tick labels, etc.
ax.set_title("Elapse = 0")
ax.set_xlabel("RPS")
ax.set_ylabel("P99 Latency (ms)")
# ax.set_ylim(0, 40)
ax.set_xticks(x)
ax.set_xticklabels(data["rps"])
ax.legend()

fig.tight_layout()
plt.savefig("ex0.pdf")
plt.show()